# Strata on Kaggle — SC117 Abliterated Qwen3.8-Flash-Next

One-click Kaggle notebook for [SC117's abliterated Qwen3.8-Flash-Next GSQ-RCO GGUF](https://huggingface.co/SC117/Qwen3.8-Flash-Next-GSQ-RCO-abliterated-GGUF), based on the DevZoneX Strata Kaggle workflow.

It keeps the upstream workflow's Kaggle secrets, hardware checks, private model/runtime cache datasets, authenticated OpenAI-compatible API, Cloudflare Quick Tunnel, end-to-end API test, and OpenCode configuration output. It replaces the built-in Gyro-S preset with an explicit SC117 configuration.

**Important model details**
- The main model must have its own native pack built from the selected SC117 GGUF shards. Do not reuse the upstream model's pack.
- The notebook downloads both shards for one selected tier and SC117's Strata-only MTP runtime.
- SC117's published MTP draft layer comes from the stock Qwen checkpoint; it is not itself abliterated.
- The first run is long: it downloads tens of gigabytes, builds the native pack, and may need to install the Strata engine. Later sessions restore private Kaggle datasets, although Kaggle still stages attached datasets into each new runtime.

## Before you run
1. Enable Kaggle Internet and a GPU accelerator.
2. Set your Kaggle username in the configuration cell.
3. Add a Kaggle Secret named `KAGGLE_API_TOKEN` containing your Kaggle API token.
4. Read the Hugging Face model card and license/terms. Set the explicit private-cache confirmation to `True` only when you agree to those terms and want the model stored in your private Kaggle Dataset.
5. Run cells from top to bottom.

Model setup notes: [SC117 Strata README](https://huggingface.co/SC117/Qwen3.8-Flash-Next-GSQ-RCO-abliterated-GGUF/blob/main/strata/README.md).

In [ ]:
# Kaggle may mount /root/.ipython read-only. Move shell DB writes to writable session storage.
from pathlib import Path
IPYTHON_STATE_DIR = Path('/kaggle/working/strata-ipython-state')
IPYTHON_STATE_DIR.mkdir(parents=True, exist_ok=True)
try:
    from pickleshare import PickleShareDB
    get_ipython().db = PickleShareDB(str(IPYTHON_STATE_DIR / 'db'))
    print('IPython shell database redirected to', IPYTHON_STATE_DIR / 'db')
except Exception as exc:
    print('IPython shell database redirect skipped:', repr(exc))
try:
    get_ipython().history_manager.enabled = False
    print('IPython SQLite history writes disabled for this session.')
except Exception as exc:
    print('IPython history-manager adjustment skipped:', repr(exc))
# Use os.chdir(...) or subprocess.run(..., cwd=...) instead of %cd if dhist writes fail.


In [ ]:
# ===== USER CONFIGURATION: EDIT THIS CELL =====
KAGGLE_USERNAME = "your-kaggle-username"
MODEL_REPO = "SC117/Qwen3.8-Flash-Next-GSQ-RCO-abliterated-GGUF"
MODEL_TIER = "Q2_0"  # Q2_0, IQ2_XS, IQ3_XXS, IQ3_S
CONTEXT = "32k"
GPU_IDS = ""  # blank = detect all visible NVIDIA GPUs
MODEL_DATA_DIR = "/kaggle/working/strata-local"  # writable session storage; caches persist via private Kaggle Datasets

# "symlink" avoids a second local copy of the large model; input-dataset reads may be slower.
# Use "copy" if local reads are too slow and disk space permits.
MODEL_RESTORE_MODE = "symlink"

import re
_slug = lambda value: re.sub(r"[^a-z0-9-]+", "-", value.lower()).strip("-")
MODEL_DATASET_SLUG = _slug(f"strata-model-sc117-{MODEL_TIER}")
RUNTIME_DATASET_SLUG = "strata-runtime-cache-sc117"

UPLOAD_CACHES = True
I_CONFIRM_MODEL_LICENSE_AND_PRIVATE_STORAGE = False
FORCE_UPDATE_MODEL_CACHE = False
FORCE_UPDATE_RUNTIME_CACHE = False

print("Model:", MODEL_REPO, MODEL_TIER)
print("Model cache:", f"{KAGGLE_USERNAME}/{MODEL_DATASET_SLUG}")
print("Runtime cache:", f"{KAGGLE_USERNAME}/{RUNTIME_DATASET_SLUG}")
print("Confirm private model caching:", I_CONFIRM_MODEL_LICENSE_AND_PRIVATE_STORAGE)
print("Restore mode:", MODEL_RESTORE_MODE)

## 1. Load this fork's helpers and Kaggle credentials

The notebook clones this fork's main branch and loads its SC117-specific model preparation helpers. The Kaggle API token is read from Kaggle Secrets and is not embedded in the notebook.

In [ ]:
import os, sys, subprocess, json, re, shutil, importlib
from pathlib import Path

REPO_BRANCH = "fix/kaggle-writable-paths-and-restart"
REPO_URL = "https://github.com/seczoid/devzonex-strata-kaggle-abliterated.git"
REPO_DIR = Path("/kaggle/working/devzonex-strata-kaggle-abliterated")
if not (REPO_DIR / "scripts" / "sc117_model.py").is_file():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    # Existing Kaggle clones may have a shallow/single-branch fetch refspec.
    # Fetch this branch explicitly, then point the local branch at the fetched commit.
    subprocess.run([
        "git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin",
        f"refs/heads/{REPO_BRANCH}:refs/remotes/origin/{REPO_BRANCH}"
    ], check=True)
    subprocess.run([
        "git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH,
        f"origin/{REPO_BRANCH}"
    ], check=True)

print("Using repository branch:", REPO_BRANCH)
print("Repository commit:", subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], check=True, capture_output=True, text=True).stdout.strip())

sys.path.insert(0, str(REPO_DIR))
from scripts import strata_kaggle as sk
from scripts import sc117_model as sm
sk = importlib.reload(sk)
sm = importlib.reload(sm)

KAGGLE_ENV = sk.kaggle_env(KAGGLE_USERNAME, secret_name="KAGGLE_API_TOKEN")
sk.install_kaggle_cli(KAGGLE_ENV)
HARDWARE = sk.hardware_summary()
print(json.dumps(HARDWARE, indent=2))
if not HARDWARE["gpu_names"]:
    raise RuntimeError("No NVIDIA GPU detected. Enable a GPU accelerator in Kaggle settings.")


## 2. Restore private caches and prepare SC117

On a first run, the model and MTP files are downloaded from Hugging Face, then the SC117-specific native pack is built. The Strata runtime is installed if no matching private runtime dataset is attached.

On returning runs, use Kaggle's **Add Input** control to attach both datasets printed in the configuration cell:
- the private SC117 model cache for your selected tier
- the private Strata runtime cache

Kaggle still stages attached dataset files into each new session. Symlink mode avoids making another local copy of the model, but does not eliminate Kaggle's own staging time. Set restore mode to `copy` if local reads from the input mount are too slow and enough disk space is available.

The cache uses the `sc117` family and tier-specific names to avoid mixing SC117 files with the upstream Gyro-S cache.

In [ ]:
MODEL_ROOT = Path(MODEL_DATA_DIR)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

RUNTIME_CACHE_RESTORED = sk.restore_runtime_cache(KAGGLE_USERNAME, RUNTIME_DATASET_SLUG)
MODEL_CACHE_RESTORED = sk.restore_model_cache(
    KAGGLE_USERNAME, MODEL_DATASET_SLUG, MODEL_ROOT, workers=8,
    family="sc117", variant=MODEL_TIER.lower(), restore_mode=MODEL_RESTORE_MODE,
)
sk.install_strata_if_missing()

print("Runtime cache restored:", RUNTIME_CACHE_RESTORED)
print("Model cache restored:", MODEL_CACHE_RESTORED)
print("Strata terminal installed and version checked.")

SC117_PATHS = sm.prepare_sc117_model(MODEL_REPO, MODEL_TIER, MODEL_ROOT)
print("SC117 model preparation complete.")


## 3. Review the SC117 custom launch plan

This is a custom Strata configuration, not the built-in `gyro / gyro-s` preset. The preflight checks both GGUF shards, the tokenizer, the model-specific native pack and the MTP runtime. If no Strata engine component is cached, it asks Strata to prepare one using the already-downloaded local SC117 files rather than fetching the stock model.

In [ ]:
if not GPU_IDS.strip():
    GPU_IDS = sk.default_gpu_list()
print("GPU selection:", GPU_IDS)
PLAN_OUTPUT = sm.run_sc117_plan(
    paths=SC117_PATHS, context=CONTEXT, gpu_list=GPU_IDS, data_dir=MODEL_ROOT,
)


## 4. Launch the authenticated SC117 server and Cloudflare tunnel

The server uses the SC117-specific pack and tokenizer, the two original GGUF shards, and the model repository's MTP runtime. A fresh API key is stored in a private local file and is not printed in the notebook output.

In [ ]:
import requests

server_is_ready = False
health_json = {}
try:
    r = requests.get("http://127.0.0.1:8080/health", timeout=5)
    health_json = r.json() if r.ok else {}
    if health_json.get("loaded"):
        current_model = str(health_json.get("model") or "")
        if "sc117" not in current_model.lower():
            raise RuntimeError(
                f"Port 8080 is already serving a different model ({current_model}). "
                "Stop that server before launching this notebook."
            )
        if not health_json.get("api_key"):
            raise RuntimeError("The existing SC117 server is not API-key protected; refusing to expose it.")
        server_is_ready = True
except RuntimeError:
    raise
except Exception:
    pass

if server_is_ready:
    print("Reusing the healthy, API-key-protected SC117 server.")
else:
    LAUNCH_OUTPUT = sm.launch_sc117(
        paths=SC117_PATHS, context=CONTEXT, gpu_list=GPU_IDS, data_dir=MODEL_ROOT,
    )
    print(LAUNCH_OUTPUT)

try:
    HEALTH = sk.wait_health(timeout_seconds=900)
except Exception:
    for log_path in (sm.SERVER_LOG, sm.ENGINE_LOG):
        if log_path.is_file():
            print(f"\\n--- Log tail: {log_path} ---")
            print(log_path.read_text(encoding="utf-8", errors="replace")[-8000:])
    raise
print("Loaded:", HEALTH.get("loaded"))
print("Model:", HEALTH.get("model"))

TUNNEL_URL = sk.start_quick_tunnel()
print("Temporary HTTPS endpoint:", TUNNEL_URL)


## 5. Verify the public API and run a short generation

The next cell checks that unauthenticated API requests are rejected with HTTP 401/403, authenticated requests can list models, and a short authenticated chat completion succeeds. It never prints the Strata API key.

In [ ]:
TEST_RESULT = sk.verify_public_endpoint(TUNNEL_URL)
MODEL_ID = TEST_RESULT["model_id"]
print("End-to-end authenticated API test: PASS")
print("Model ID for OpenCode:", MODEL_ID)

## 6. Save your personal private caches

The model and runtime are stored in separate datasets owned by your Kaggle account. Dataset upload is disabled for the model unless you explicitly confirm the model's license permits your private copy. The uploader never passes a public-visibility flag. Kaggle may need time to process datasets after the upload finishes.

In [ ]:
if UPLOAD_CACHES:
    MODEL_UPLOADED = sk.upload_model_cache(
        data_dir=MODEL_ROOT,
        username=KAGGLE_USERNAME,
        slug=MODEL_DATASET_SLUG,
        family="sc117",
        variant=MODEL_TIER.lower(),
        env=KAGGLE_ENV,
        license_confirmed=I_CONFIRM_MODEL_LICENSE_AND_PRIVATE_STORAGE,
        force=FORCE_UPDATE_MODEL_CACHE,
    )
    RUNTIME_UPLOADED = sk.upload_runtime_cache(
        username=KAGGLE_USERNAME,
        slug=RUNTIME_DATASET_SLUG,
        env=KAGGLE_ENV,
        force=FORCE_UPDATE_RUNTIME_CACHE,
    )
    print("Model cache upload submitted:", MODEL_UPLOADED)
    print("Runtime cache upload submitted:", RUNTIME_UPLOADED)
    print("Wait for Kaggle dataset processing to finish before ending the session.")
else:
    print("Cache uploads disabled by configuration.")


## 7. Connect OpenCode Desktop

Merge the printed provider entry into your existing OpenCode configuration; do not replace unrelated providers. Retrieve the Strata API key privately in the Kaggle shell with `strata launch --show-key`. Save it locally on your Windows PC at `~/.config/opencode/strata-key.txt`. Never commit or share the key.

Keep this Kaggle session and tunnel running while using the model. The URL may change after a restart.

In [ ]:
print("OpenCode baseURL:", TUNNEL_URL.rstrip("/") + "/v1")
print("OpenCode model ID:", MODEL_ID)
print("\nMerge this provider entry inside the existing top-level providers object:")
print(sk.opencode_config_snippet(TUNNEL_URL, MODEL_ID))
print("\nAPI key is stored in this private local file:", sm.API_KEY_FILE)
print("When you need to copy it, run this in a private Kaggle cell and do not share the output:")
print("print(sm.API_KEY_FILE.read_text().strip())")


## 8. Keep the notebook actively supervising Strata (optional, 6 hours)

Run this cell after the API test succeeds. It keeps an actual health-monitoring task running while checking the local Strata service and the tunnel's authentication boundary. It does **not** bypass Kaggle's maximum runtime or guarantee uptime; the session, GPU quota, network, and tunnel can still be stopped by Kaggle or fail. Stop this cell when you're done.

In [ ]:
import time
from datetime import datetime, timezone

# Set to 6 for a six-hour supervision window. Do not exceed Kaggle's session limits.
MONITOR_HOURS = 6
HEALTH_CHECK_SECONDS = 60
REPORT_EVERY_SECONDS = 300

monitor_started = time.monotonic()
monitor_deadline = monitor_started + MONITOR_HOURS * 3600
last_report = 0.0
last_state = None
print(f"Monitoring Strata for up to {MONITOR_HOURS} hours. Keep Kaggle session within its runtime/GPU quota.", flush=True)

while time.monotonic() < monitor_deadline:
    now_mono = time.monotonic()
    remaining = max(0, int(monitor_deadline - now_mono))
    try:
        response = requests.get("http://127.0.0.1:8080/health", timeout=10)
        payload = response.json() if response.ok else {}
        healthy = bool(response.ok and payload.get("status") == "ok" and payload.get("loaded") is True and payload.get("api_key") is True)
        state = "HEALTHY" if healthy else f"NOT READY (HTTP {response.status_code}; loaded={payload.get('loaded')}; api_key={payload.get('api_key')})"
    except Exception as exc:
        healthy = False
        state = "HEALTH CHECK ERROR: " + str(exc)[:180]

    # Occasionally make sure the public endpoint is still reachable and rejects no-key requests.
    if healthy and (now_mono - last_report >= REPORT_EVERY_SECONDS or last_state != state):
        try:
            public_check = requests.get(TUNNEL_URL.rstrip("/") + "/v1/models", timeout=15)
            if public_check.status_code not in (401, 403):
                state += f" | WARNING: public no-key check returned HTTP {public_check.status_code} (expected 401/403)"
                healthy = False
            else:
                state += f" | tunnel/auth boundary OK (HTTP {public_check.status_code} without key)"
        except Exception as exc:
            state += " | tunnel check failed: " + str(exc)[:160]
            healthy = False

    if now_mono - last_report >= REPORT_EVERY_SECONDS or state != last_state:
        print(f"[{datetime.now().astimezone().isoformat(timespec='seconds')}] {state} | {remaining//3600:02d}:{(remaining%3600)//60:02d}:{remaining%60:02d} remaining", flush=True)
        last_report = now_mono
        last_state = state
    time.sleep(HEALTH_CHECK_SECONDS)

print("Supervision window complete. Strata may continue until Kaggle stops the session; check health before use.", flush=True)

> **Important:** A running monitor cell is a legitimate health monitor, not an uptime guarantee. Kaggle may end a session at its hard limit, when quotas are exhausted, or if the runtime/server fails. For a reliable always-on endpoint, use infrastructure designed for persistent services.